In [12]:
import torch
from torch.utils import data
from torch.utils.data import DataLoader
from torch import nn
import torch.nn.functional as F

import random
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import scipy
from tqdm import trange
from tqdm import tqdm
from datetime import datetime
import sys
import os
import time
from sklearn.metrics import roc_auc_score, f1_score
from IPython.display import clear_output

from transformers import BertModel, BertConfig, PreTrainedTokenizer, BasicTokenizer, BertForTokenClassification, utils

import sklearn
from sklearn.metrics import accuracy_score

from Bio import SeqIO
from io import StringIO, BytesIO

import gc

import warnings
warnings.filterwarnings("ignore")

In [ ]:
from dna_tokenizer import DNATokenizer, seq2kmer

In [ ]:
from tabulate import tabulate
def to_fwf(df, fname):
    content = tabulate(df.values.tolist(), tablefmt="plain")
    open(fname, "w").write(content)

pd.DataFrame.to_fwf = to_fwf

In [ ]:
def split_seq(seq, length = 512, pad = 16):
    res = []
    n = len(seq)
    for st in range(0, n, length - pad):
        if st > 0 and st + pad >= n:
            break
        end = min(st+length, n)
        res.append(seq[st:end])
    return res

In [ ]:
def stitch_np_seq(np_seqs, pad=16):
    # Calculate total length first to pre-allocate array
    total_length = sum(seq.shape[-1] for seq in np_seqs) - pad * (len(np_seqs) - 1)
    
    # Pre-allocate result array
    res = np.empty(total_length, dtype=np_seqs[0].dtype)
    
    # Track current position
    pos = 0
    
    for i, seq in enumerate(np_seqs):
        seq_len = seq.shape[-1]
        if i > 0:
            # Overlap by removing 'pad' elements from previous segment
            pos -= pad
        # Copy current sequence into the result
        #print(pos,pos+seq_len)
        res[pos:pos+seq_len] = seq[0,:]
        pos += seq_len
    
    return res

In [ ]:
class PredDataset(data.Dataset):
    def __init__(self, sequence, tokenizer):
        self.pieces = split_seq(seq2kmer(sequence.upper(),6).split(' '), length = 512, pad = 16)
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.pieces)
    
    def __getitem__(self, index):
        sequence = self.pieces[index]
        encoded_k_mers = self.tokenizer.encode_plus(sequence, add_special_tokens=False, max_length=512)["input_ids"]
        return torch.LongTensor(encoded_k_mers)

In [ ]:
tokenizer = DNATokenizer.from_pretrained('6-new-12w-0/')

In [2]:
import pickle
import gc

pickle_dir = 'predictions'

def get_pickle_filename(name):
    """Determine the appropriate pickle filename based on the name pattern"""
    if name.startswith('NW_'):
        return os.path.join(pickle_dir, 'NW__all.pickle')
    else:
        return os.path.join(pickle_dir, f'{name}.pickle')
    
def name_already_processed(name):
    """Check if a name has already been processed by looking for its pickle file"""    
    if os.path.exists(get_pickle_filename(name)):
        if name.startswith('NW_'):
            try:
                with open(pickle_file, 'rb') as f:
                    existing_data = pickle.load(f)
                    return name in existing_data
            except:
                return False
        return True
    return False

In [5]:
import numpy as np
import scipy.ndimage

def extract_segments_1d(arr, name, threshold, min_len,
                        chr_names, starts, ends, scores):
    mask = arr > threshold
    labeled, n = scipy.ndimage.label(mask)

    if n == 0:
        return

    # For 1D labels, each object is returned as a tuple like (slice(start, stop),)
    objects = scipy.ndimage.find_objects(labeled)

    for slc_tuple in objects:
        if slc_tuple is None:
            continue
        slc = slc_tuple[0]
        start, end = slc.start, slc.stop

        if end - start > min_len:
            seg_score = float(arr[start:end].mean())
            chr_names.append(name)
            starts.append(start)
            ends.append(end)
            scores.append(seg_score)

In [6]:
from tqdm import trange
model_confidence_threshold = 0.25
min_len = 4

chr_names, starts, ends, scores = [], [], [], []

for fname in os.listdir('predictions/'):
    print(fname)
    with open('predictions/' + fname,'rb') as f:
        preds = pickle.load(f)
        
    if fname.startswith('NW_'):
        for k, arr in preds.items():
            print(k, arr.shape)
            extract_segments_1d(
            arr=arr,
            name=k,
            threshold=model_confidence_threshold,
            min_len=min_len,
            chr_names=chr_names,
            starts=starts,
            ends=ends,
            scores=scores,
        )
    else:
        print(preds.shape)
        extract_segments_1d(
        arr=preds,
        name=fname.replace('.pickle', ''),
        threshold=model_confidence_threshold,
        min_len=min_len,
        chr_names=chr_names,
        starts=starts,
        ends=ends,
        scores=scores,
    )


NC_068981.1.pickle
(199874324,)
NC_068982.1.pickle
(192492725,)
NC_068983.1.pickle
(168056172,)
NC_068984.1.pickle
(147805811,)
NC_068985.1.pickle
(127476054,)
NC_068986.1.pickle
(117080274,)
NC_068987.1.pickle
(110520654,)
NC_068988.1.pickle
(97793168,)
NC_068989.1.pickle
(96881191,)
NC_068990.1.pickle
(94598369,)
NC_068991.1.pickle
(80868366,)
NC_068992.1.pickle
(68459955,)
NC_068993.1.pickle
(65066832,)
NC_068994.1.pickle
(60576129,)
NC_068995.1.pickle
(58155570,)
NC_068996.1.pickle
(57524426,)
NC_068997.1.pickle
(57421545,)
NC_068998.1.pickle
(55557499,)
NC_068999.1.pickle
(55137724,)
NC_069000.1.pickle
(47890966,)
NC_069001.1.pickle
(40062745,)
NC_069002.1.pickle
(37717994,)
NC_069003.1.pickle
(37682606,)
NC_069004.1.pickle
(36166158,)
NC_069005.1.pickle
(35489831,)
NC_069006.1.pickle
(26644802,)
NC_069007.1.pickle
(22553644,)
NC_069008.1.pickle
(18637140,)
NC_069009.1.pickle
(11991808,)
NC_069010.1.pickle
(10083076,)
NC_029723.1.pickle
(15728,)
NW__all.pickle
NW_026294888.1 (344,

In [7]:
df = pd.DataFrame(
    list(zip(chr_names, starts, ends, scores)),
    columns=["chrom", "start", "end", "score"]
)

In [11]:
out_path = 'GCF_001194135.2_ASM119413v2_zdna_thr025.bedgraph'
# Optional track line (UCSC/IGV will ignore if not needed)
track = 'track type=bedGraph name="ZDNA_thr0.25" description="mean score per segment" visibility=full autoScale=on'
with open(out_path, 'w') as out:
    out.write(track + '\n')
    df[["chrom", "start", "end", "score"]].to_csv(out, sep='\t', header=False, index=False)
print(f"Wrote BedGraph: {out_path}")

Wrote BedGraph: GCF_001194135.2_ASM119413v2_zdna_thr025.bedgraph


In [13]:
from tqdm import trange
model_confidence_threshold = 0.25
min_len = 4

chr_names, starts, ends, scores = [], [], [], []

for fname in os.listdir('predictions_g4/'):
    print(fname)
    with open('predictions_g4/' + fname,'rb') as f:
        preds = pickle.load(f)
        
    if fname.startswith('NW_'):
        for k, arr in preds.items():
            print(k, arr.shape)
            extract_segments_1d(
            arr=arr,
            name=k,
            threshold=model_confidence_threshold,
            min_len=min_len,
            chr_names=chr_names,
            starts=starts,
            ends=ends,
            scores=scores,
        )
    else:
        print(preds.shape)
        extract_segments_1d(
        arr=preds,
        name=fname.replace('.pickle', ''),
        threshold=model_confidence_threshold,
        min_len=min_len,
        chr_names=chr_names,
        starts=starts,
        ends=ends,
        scores=scores,
    )


NC_068981.1.pickle
(199874324,)
NC_068982.1.pickle
(192492725,)
NC_068983.1.pickle
(168056172,)
NC_068984.1.pickle
(147805811,)
NC_068985.1.pickle
(127476054,)
NC_068986.1.pickle
(117080274,)
NC_068987.1.pickle
(110520654,)
NC_068988.1.pickle
(97793168,)
NC_068989.1.pickle
(96881191,)
NC_068990.1.pickle
(94598369,)
NC_068991.1.pickle
(80868366,)
NC_068992.1.pickle
(68459955,)
NC_068993.1.pickle
(65066832,)
NC_068994.1.pickle
(60576129,)
NC_068995.1.pickle
(58155570,)
NC_068996.1.pickle
(57524426,)
NC_068997.1.pickle
(57421545,)
NC_068998.1.pickle
(55557499,)
NC_068999.1.pickle
(55137724,)
NC_069000.1.pickle
(47890966,)
NC_069001.1.pickle
(40062745,)
NC_069002.1.pickle
(37717994,)
NC_069003.1.pickle
(37682606,)
NC_069004.1.pickle
(36166158,)
NC_069005.1.pickle
(35489831,)
NC_069006.1.pickle
(26644802,)
NC_069007.1.pickle
(22553644,)
NC_069008.1.pickle
(18637140,)
NC_069009.1.pickle
(11991808,)
NC_069010.1.pickle
(10083076,)
NC_029723.1.pickle
(15728,)
NW__all.pickle
NW_026294888.1 (344,

In [15]:
df = pd.DataFrame(
    list(zip(chr_names, starts, ends, scores)),
    columns=["chrom", "start", "end", "score"]
)

In [16]:
out_path = 'GCF_001194135.2_ASM119413v2_G4_thr025.bedgraph'
# Optional track line (UCSC/IGV will ignore if not needed)
track = 'track type=bedGraph name="G4_thr0.25" description="mean score per segment" visibility=full autoScale=on'
with open(out_path, 'w') as out:
    out.write(track + '\n')
    df[["chrom", "start", "end", "score"]].to_csv(out, sep='\t', header=False, index=False)
print(f"Wrote BedGraph: {out_path}")

Wrote BedGraph: GCF_001194135.2_ASM119413v2_G4_thr025.bedgraph


In [17]:
df

,chrom,start,end,score
0,NC_068981.1,96210,96224,0.786792
1,NC_068981.1,139872,139885,0.426565
2,NC_068981.1,140203,140222,0.516117
3,NC_068981.1,291548,291556,0.280562
4,NC_068981.1,382075,382093,0.699734
...,...,...,...,...
38527,NW_026439921.1,1984,1998,0.365590
38528,NW_026439958.1,0,9,0.280401
38529,NW_026440011.1,195,201,0.467667
38530,NW_026440030.1,103,130,0.559187


In [29]:
print(len(starts))
print(len(ends))
print(len(chr_names))

264865
264865
264865


In [32]:
df = pd.DataFrame(list(zip(chr_names, starts, ends)))

In [47]:
#[t for t in list(set(df[0].tolist())) if 'Eber_ch' in t]

In [ ]:
#Eber_sc12555

In [44]:
df.sort_values([0])

,0,1,2
100630,Eber_ch1,19923,19928
105887,Eber_ch1,112364985,112364990
105886,Eber_ch1,112293044,112293060
105885,Eber_ch1,112086886,112086899
105884,Eber_ch1,112086743,112086760
...,...,...,...
188656,Eber_sc9996,17388,17396
188658,Eber_sc9996,23287,23300
188660,Eber_sc9998,3569,3578
188659,Eber_sc9998,3482,3502


In [46]:
df.sort_values([0]).to_fwf(f'Eber_ch1-48_sc1-1255_thr05_hg.bed')